# 06 — Decorators

`🟡 Intermediate`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/02_Intermediate/06_decorators.ipynb)

## 📌 What is it?
A decorator is a function that wraps another function to add behavior without changing its source code. The `@` syntax is syntactic sugar.

## 🧠 Why AI Engineers need this
Decorators power: caching, logging, retry logic, rate limiting, timing, authentication, and more. Many ML libraries use them extensively.

In [ ]:
# ── HOW DECORATORS WORK ──
import time
from functools import wraps

def timer(func):
    """Decorator that measures execution time."""
    @wraps(func)   # preserves func's name and docstring!
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱️ {func.__name__}() took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def load_model(name):
    """Simulate loading a model."""
    time.sleep(0.1)   # simulate load time
    return f"{name} loaded"

result = load_model("bert-base-uncased")
print(result)

In [ ]:
# ── DECORATOR WITH ARGUMENTS ──
from functools import wraps
import time

def retry(max_attempts=3, delay=1.0, exceptions=(Exception,)):
    """Retry decorator with configurable attempts and delay."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        print(f"All {max_attempts} attempts failed")
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.1, exceptions=(ValueError,))
def unstable_api_call(n):
    if n < 2:
        raise ValueError("Not ready yet")
    return "Success!"

call_count = 0
def test():
    global call_count
    call_count += 1
    return unstable_api_call(call_count)

print(test())

In [ ]:
# ── STACKING DECORATORS ──
from functools import wraps
import time

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📞 Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"✅ {func.__name__} returned: {result}")
        return result
    return wrapper

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"⏱️ {func.__name__}: {time.perf_counter()-start:.4f}s")
        return result
    return wrapper

@log_call
@timer   # decorators applied bottom-up
def generate_text(prompt):
    time.sleep(0.05)
    return f"Generated response for: {prompt[:20]}"

generate_text("What is machine learning?")

## ⚠️ Common Mistakes
```python
# ❌ Forgetting @wraps — breaks introspection
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def my_func():
    """My docstring"""
    pass

print(my_func.__name__)   # 'wrapper' — wrong!
print(my_func.__doc__)    # None — docstring lost!

# ✅ Always use @wraps(func)
```

## 🔗 What's Next?
[07 — Generators & Iterators →](07_generators_and_iterators.ipynb)